### **Classification de sentiments sur des critiques de films**

In [1]:
from src.transformer_modules.transformer_modules import *
from src.modeles.transformer_classifier import TransformerClassifier
import urllib.request
from loguru import logger
import tarfile
import os
import re
from collections import Counter
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torch import nn

**Chargement des données**


Il s'agit d'un jeu de données pour la classification binaire des sentiments. Il y a 25 000 critiques de films très polarisées pour l'entraînement et 25 000 pour les tests.

In [ ]:
url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
output = "data/aclImdb_v1.tar.gz"

if not os.path.exists(output):
    urllib.request.urlretrieve(url, output)

if not os.path.exists("data/aclImdb"):
    with tarfile.open(output, "r:gz") as f:
        f.extractall(path="data")
    logger.info(f"Extracted {output}")

**Préparation des données**

In [ ]:
def clean_text(text):
    """Nettoyage des données"""
    text = text.lower()
    text = re.sub(r"[^a-z'àâäéèêëîïôöùûüç0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

#-------------------------------------------------

def create_vocab(texts, vocab_size):
    """Crée un vocabulaire à partir des textes"""
    counter = Counter()
    
    for text in texts:
        cleaned = clean_text(text)
        tokens = cleaned.split()
        counter.update(tokens)
    
    # Prendre les mots les plus fréquents
    most_common = counter.most_common(vocab_size-2)
    
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for i, (word, _) in enumerate(most_common):
        vocab[word] = i + 2
    
    return vocab

#-------------------------------------------------

def create_padding_mask(x, pad_idx=0):
    return (x != pad_idx).unsqueeze(1).unsqueeze(2)
   

#-------------------------------------------------

def load_and_prepare_imdb(path, vocab = None, vocab_size=20000, max_len=256):
    """Charge les données et prépare pour le Transformer"""
    texts = []
    labels = []
    
    #  Charger les textes bruts
    for label, folder in [(1, "pos"), (0, 'neg')]:
        folder_path = os.path.join(path, folder)
        
        for filename in os.listdir(folder_path):
            with open(os.path.join(folder_path, filename), "r", encoding="utf-8") as f:
                text = f.read()
                texts.append(text)
                labels.append(label)
    
    #  Créer le vocabulaire
    if vocab is None:
       vocab = create_vocab(texts, vocab_size)
    
    #  Convertir textes en indices
    encoded_texts = []
    for text in texts:
        # Nettoyer
        text = clean_text(text)
        # Tokenizer
        tokens = text.split()[:max_len]
        # Convertir en indices
        indices = [vocab.get(token, 1) for token in tokens]  # 1 = <UNK>
        # Padding
        indices+= [vocab["<PAD>"]] * (max_len - len(indices))  # 0 = <PAD>
        encoded_texts.append(indices)

    return torch.tensor(encoded_texts), torch.tensor(labels), vocab

In [ ]:
# préparation des données de train 
max_len = 256
X_train, y_train, vocab = load_and_prepare_imdb(
      "data/aclImdb/train",
      vocab=None, 
      max_len=max_len
)

# Création des DataLoaders
batch_size = 32
train_dataset = TensorDataset(X_train, y_train.float())
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Preparation des données de validation et de test
X_val_all, y_val_all, _ = load_and_prepare_imdb("data/aclImdb/test", vocab=vocab)
X_val, X_test, y_val, y_test = train_test_split(
        X_val_all,
        y_val_all,
        test_size=0.2, 
        random_state=142,
        stratify=y_val_all
)

val_dataset = TensorDataset(X_val, y_val.float())
test_dataset = TensorDataset(X_test, y_test.float()) 

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
X_train.shape, y_train.shape,  X_test.shape, y_test.shape, X_val.shape, y_val.shape

In [ ]:
#  Configuration du device (CPU ou GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

**Fonctions d'entrainement et d'évaluation**

In [ ]:
import torch.optim as optim
from torch.nn import BCEWithLogitsLoss
from tqdm import tqdm

# ------------------ Fonction d'entraînement du modèle -------------------
def train(model, loader, optimizer, criterion, device, use_mask=True): # 0 signifie pas de masque
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_X, batch_y in tqdm(loader, desc="Training"):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        mask = create_padding_mask(batch_X).to(device) if use_mask else None
        
        optimizer.zero_grad()
        outputs = model(batch_X, mask)
        
        # Normaliser la forme de la sortie en un logit par exemple
        # Cas attendus: (batch, 1) -> squeeze -> (batch,), ou (batch,) déjà
        if outputs.dim() == 3:
            logits = outputs[:, 0, :]
            if logits.size(1) == 1:
                logits = logits.squeeze(1)
        else:
            logits = outputs.squeeze(-1)
        
        loss = criterion(logits, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        predictions = (torch.sigmoid(logits) > 0.5).float()
        correct += (predictions == batch_y).sum().item()
        total += batch_y.size(0)
    
    return total_loss / len(loader), correct / total

# ---------------- Fonction d'évaluation du modèle -------------------
def evaluate(model, loader, criterion, device, use_mask=True): # 0 signifie pas de masque  
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in tqdm(loader, desc="Evaluating"):
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            mask = create_padding_mask(batch_X).to(device) if use_mask else None
            
            outputs = model(batch_X, mask)

            if outputs.dim() == 3:
                logits = outputs[:, 0, :]
                if logits.size(1) == 1:
                    logits = logits.squeeze(1)
            else:
                logits = outputs.squeeze(-1)

            loss = criterion(logits, batch_y)
            total_loss += loss.item()
            
            predictions = (torch.sigmoid(logits) > 0.5).float()
            correct += (predictions == batch_y).sum().item()
            total += batch_y.size(0)
    
    return total_loss / len(loader), correct / total

# ---------------- Fonction  d'ntraînement du modèle avec 10 époques -------------------
def train_model(model, train_loader, test_loader, optimizer, criterion, device, epochs):
    train_losses = []
    train_accs = []
    test_losses = []
    test_accs = []
    
    
    criterion = BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2)

    train_losses, test_losses = [], []
    train_accs, test_accs = [], []

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        # Entraînement
        train_loss, train_acc = train(model, train_loader, optimizer, criterion, device, use_mask=False)
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Validation
        test_loss, test_acc = evaluate(model, test_loader, criterion, device, use_mask=False)
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        
        scheduler.step(test_loss)
        
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {test_loss:.4f}, Val Acc: {test_acc:.4f}")

In [ ]:
import time
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Fonction  d'évaluation des métriques avec temps d'évaluation sur l'ensemble de test
@torch.no_grad()
def test_metrics_with_time(model, loader, device, use_mask=True, threshold=0.5):
    """
    Retourne: test_acc, test_f1, test_auc, eval_time_sec
    """
    model.eval()

    all_logits = []
    all_y = []

    start = time.time()

    for batch_X, batch_y in loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        batch_mask = create_padding_mask(batch_X).to(device) if use_mask else None
        logits = model(batch_X, batch_mask).squeeze(-1)  # (B,)

        all_logits.append(logits.detach().cpu())
        all_y.append(batch_y.detach().cpu())

    eval_time = time.time() - start

    logits = torch.cat(all_logits).numpy()
    y_true = torch.cat(all_y).numpy()

    y_prob = 1.0 / (1.0 + np.exp(-logits))           # sigmoid
    y_pred = (y_prob >= threshold).astype(np.float32)

    test_acc = accuracy_score(y_true, y_pred)
    test_f1  = f1_score(y_true, y_pred, zero_division=0)
    test_auc = roc_auc_score(y_true, y_prob)

    return test_acc, test_f1, test_auc, eval_time

# --------------- Fonction d'éxcution du modèle ----------------------------------
def run_model_and_collect(model, name, train_loader, dev_loader, test_loader,
                          optimizer, criterion, device, epochs=10, use_mask=True):
    """
    - Entraîne 'epochs' epochs
    - (option) utilise dev_loader juste pour afficher, pas obligatoire
    - Évalue sur test_loader et renvoie les métriques + temps
    """
    train_start = time.time()

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train(model, train_loader, optimizer, criterion, device, use_mask=use_mask)
        dev_loss, dev_acc = evaluate(model, dev_loader, criterion, device, use_mask=use_mask)
        print(f"[{name}] ====== Epoch {epoch}/{epochs} ====== | train_acc={tr_acc:.4f} | dev_acc={dev_acc:.4f}")

    train_time = time.time() - train_start

    test_acc, test_f1, test_auc, test_eval_time = test_metrics_with_time(
        model, test_loader, device, use_mask=use_mask
    )

    return {
        "Model":name,
        "Test Acc": np.round(test_acc,3),
        "Test F1": np.round(test_f1,3),
        "Test ROC-AUC": np.round(test_auc,3),
        "Train Time (s)": np.round(train_time,3),
        "Test Eval Time (s)": np.round(test_eval_time,3)
    }


**modèle transformer**

In [ ]:
results = []
criterion = torch.nn.BCEWithLogitsLoss()
epochs = 15

In [ ]:
# Initialisation du modèle avec hyperparamètres raisonnables
# On crée d'abord l'encodeur puis on le place dans un classificateur qui
# transforme la sortie en un logit par exemple (pour BCEWithLogits)
transformer = TransformerEncoderModel(
    vocab_size=len(vocab),
    d_model=256,      # Dimension des embeddings
    N=6,              # 6 blocs encodeurs
    h=8,              # 8 têtes d'attention
    d_ff=4*256,         # Dimension feed-forward
    dropout=0.5       # Dropout pour régularisation
).to(device)


transformer_model = TransformerClassifier(
    transformer, 
    d_model=256, 
    num_classes=1, 
    pool_strategy="mean", 
    dropout=0.2
).to(device)

print(f"Nombre de paramètres: {sum(p.numel() for p in transformer_model.parameters()):,}")

opt_tr = torch.optim.AdamW(transformer_model.parameters(), lr=3e-4, weight_decay=1e-2)
results.append(run_model_and_collect(
    transformer_model, "Transformer",
    train_loader, test_loader, val_loader,   # test final = test_loader, dev = val_loader
    opt_tr, criterion, device,
    epochs=epochs, use_mask=True
))

**modèle GRU**

In [ ]:
from src.modeles.rnn_classifier import RNNClassifier

gru_model = RNNClassifier(
    vocab_size = len(vocab),
    embed_dim = 256,
    hidden_dim = 128,
    num_layers = 3,
    bidirectional = True,
    dropout = 0.2,
    pad_idx = 0,
    rnn_type = "gru"
).to(device)

print(f"Nombre de paramètres: {sum(p.numel() for p in gru_model.parameters()):,}")
#train_model(gru_model,train_loader,test_loader,optimizer=None,criterion=None,device=device,epochs=10)

opt_gru = torch.optim.AdamW(gru_model.parameters(), lr=3e-4, weight_decay=1e-2)
results.append(run_model_and_collect(
    gru_model, "GRU",
    train_loader, test_loader, val_loader,
    opt_gru, criterion, device,
    epochs=epochs, use_mask=True
))

**modèle LSTM**

In [ ]:
lstm_model = RNNClassifier(
    vocab_size = len(vocab),
    embed_dim = 256,
    hidden_dim = 128,
    num_layers = 3,
    bidirectional = True,
    dropout = 0.2,
    pad_idx = 0,
    rnn_type = "lstm"
).to(device)

print(f"Nombre de paramètres LSTM: {sum(p.numel() for p in lstm_model.parameters()):,}")

#train_model(gru_model, train_loader, test_loader, None, None, device, epochs=10)

opt_lstm = torch.optim.AdamW(lstm_model.parameters(), lr=3e-4, weight_decay=1e-2)
results.append(run_model_and_collect(
    lstm_model, "LSTM",
    train_loader, test_loader, val_loader,
    opt_lstm, criterion, device,
    epochs=epochs, use_mask=True
))

**modèle CNN**

In [ ]:
from src.modeles.cnn_classifier import TextCNNClassifier
cnn_model = TextCNNClassifier(
    vocab_size=len(vocab),
    embed_dim=128,
    kernel_sizes=[3,4,5],
    num_filters=100,
    dropout=0.2,
    pad_idx=0
).to(device)

print(f"Nombre de paramètres CNN: {sum(p.numel() for p in cnn_model.parameters()):,}")

#train_model(cnn_model, train_loader, test_loader, None, None, device, epochs=10)

opt_cnn = torch.optim.AdamW(cnn_model.parameters(), lr=3e-4, weight_decay=1e-2)
results.append( run_model_and_collect(
    cnn_model,"TextCNN",
    train_loader,val_loader,test_loader,
    optimizer=opt_cnn,
    criterion=criterion,
    device=device,
    epochs=epochs,
    use_mask=False
))

***Tableau récaputilatif des résulats des differents modèles***

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df

In [ ]:
# Sauvegarder le modèle Transformer
os.makedirs("model_save", exist_ok=True)
torch.save(transformer_model.state_dict(), "model_save/transformer_sentiment.pt")


In [ ]:
# Recharger le modèle 
model = TransformerClassifier(
    transformer, 
    d_model=256, 
    num_classes=1, 
    pool_strategy="mean", 
    dropout=0.2
).to(device)  
model.load_state_dict(torch.load("model_save/transformer_sentiment.pt", map_location="cpu"))
model.eval()
